In [ ]:
# -------------------------------------
# 01 문제정의
# -------------------------------------

In [ ]:
# -------------------------------------
# 02 데이터가져오기
# -------------------------------------

In [23]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
X = data.data     # 30개 세포 측정 특성
y = data.target   # 0=악성, 1=양성
print('X:',X.shape)
print('target_names:', list(data.target_names))

X: (569, 30)
target_names: [np.str_('malignant'), np.str_('benign')]


In [ ]:
# -------------------------------------
# 03 EDA
# -------------------------------------

In [24]:
print(y.value_counts())

target
1    357
0    212
Name: count, dtype: int64


In [25]:
X[['mean area', 'mean smoothness']].describe().loc[['min', 'max']].round(4)

,mean area,mean smoothness
min,143.5,0.0526
max,2501.0,0.1634


In [31]:
print('결측:', X.isnull().sum().sum())

결측: 0


In [ ]:
# -------------------------------------
04 데이터전처리
# -------------------------------------

In [32]:
print('X:', X.shape, '| y:', y.shape)

X: (569, 30) | y: (569,)


In [33]:
# -------------------------------------
# 05 검증데이터 분할 및 학습
# -------------------------------------

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 05-1) 분할
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)

# 05-2) 스케일링 - fit은 train에만
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_val_s = sc.transform(X_val)

# 05-3) 학습
model = LogisticRegression(max_iter=5000).fit(X_tr_s, y_tr)
print(X_tr.shape, X_val.shape)

(455, 30) (114, 30)


In [ ]:
# -------------------------------------
# 06 테스트 및 검증지표 확인
# -------------------------------------

In [35]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

pred = model.predict(X_val_s)
proba = model.predict_proba(X_val_s)[:, 1]
print('정확도 :', round(accuracy_score(y_val, pred), 4))   # 0.9825
print('ROC-AUC:', round(roc_auc_score(y_val, proba), 4))   # 0.9957
print()
print(classification_report(y_val, pred, target_names=['악성(0)', '양성(1)']))

정확도 : 0.9825
ROC-AUC: 0.9957

              precision    recall  f1-score   support

       악성(0)       1.00      0.95      0.98        42
       양성(1)       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [36]:
cm = confusion_matrix(y_val, pred)
print(cm)
print()
print('악성인데 양성이라 놓친 수:', cm[0, 1])
print('양성인데 악성이라 오진한 수:', cm[1, 0])

[[40  2]
 [ 0 72]]

악성인데 양성이라 놓친 수: 2
양성인데 악성이라 오진한 수: 0


In [37]:
# -------------------------------------
# 07 Model 내보내기
# -------------------------------------

In [38]:
import joblib

joblib.dump(model, 'model_cancer.pkl')
joblib.dump(sc, 'scaler_cancer.pkl')   # 스케일러도 반드시 같이

lm = joblib.load('model_cancer.pkl')
ls = joblib.load('scaler_cancer.pkl')
print('재로드 예측 일치:', (lm.predict(ls.transform(X_val)) == pred).all())

재로드 예측 일치: True


In [39]:
# -------------------------------------
# 08 기타
# -------------------------------------

In [41]:
# 08-1
from sklearn.metrics import recall_score

for t in [0.5, 0.9]:
    pt = (proba >= t).astype(int)
    c = confusion_matrix(y_val, pt)
    print('임계값 %.1f -> 놓친 악성 %d명 | 정확도 %.4f | 악성 recall %.4f'
          % (t, c[0, 1], accuracy_score(y_val, pt), recall_score(y_val, pt, pos_label=0)))

임계값 0.5 -> 놓친 악성 2명 | 정확도 0.9825 | 악성 recall 0.9524
임계값 0.9 -> 놓친 악성 1명 | 정확도 0.9386 | 악성 recall 0.9762


In [42]:
# 08-2
import warnings
warnings.filterwarnings('ignore')

m_raw = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)
print('[스케일링 O] 정확도 %.4f | AUC %.4f'
      % (accuracy_score(y_val, pred), roc_auc_score(y_val, proba)))
print('[스케일링 X] 정확도 %.4f | AUC %.4f'
      % (accuracy_score(y_val, m_raw.predict(X_val)), roc_auc_score(y_val, m_raw.predict_proba(X_val)[:, 1])))

[스케일링 O] 정확도 0.9825 | AUC 0.9957
[스케일링 X] 정확도 0.9474 | AUC 0.9924


In [43]:
# 08-3

In [44]:
coef = pd.Series(model.coef_[0], index=X.columns)
print(coef.abs().sort_values(ascending=False).head(5).round(3))

radius error           1.325
worst radius           0.984
worst texture          0.973
mean concave points    0.963
worst area             0.867
dtype: float64
